In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from modules.data_loader import *

In [ ]:
visco = pd.read_csv('../data/Viscoelastic_WLM.csv')
visco.head()

## Renaming columns

In [ ]:
d = {
    'Source': 'source',
    'ID_surfactant1' : 'id_surfactant1',
    'Type_surfactant1' : 'type_surfactant1',
    'SMILES_surfactant1' : 'smiles_surfactant1',
    'C_surfactant1 (mM)' : 'c_surfactant1',
    'ID_surfactant2' : 'id_surfactant2',
    'Type_surfactant2' : 'type_surfactant2',
    'SMILES_surfactant2' : 'smiles_surfactant2',
    'C_surfactant2 (mM)' : 'c_surfactant2',
    'ID_additive1' : 'id_additive1',
    'SMILES_additive1' : 'smiles_additive1',
    'C_additive1 (mM)' : 'c_additive1',
    'ID_additive2' : 'id_additive2',
    'SMILES_additive2' : 'smiles_additive2',
    'C_additive2 (mM)' : 'c_additive2',
    'T [°C]' : 'temperature',
    'Viscoelasticity' : 'viscoelasticity',
    'G₀  [Pa]' : 'g_0',
    'G"min [Pa]' : 'g_min',
    'ω* cr [rad/s]' : 'omega_cr',
    'ͳR [s]' : 'tau_r',
    'η₀[Pa∙s]' : 'eta_0'
}

visco = visco.rename(columns = d)

## Replacing NaNs

In [ ]:
visco = visco.dropna(subset=['viscoelasticity'])

In [ ]:
visco[['c_surfactant1', 'c_surfactant2', 'c_additive1', 'c_additive2']] = visco[['c_surfactant1', 'c_surfactant2', 'c_additive1', 'c_additive2']].fillna(0)

In [ ]:
visco.info()

In [ ]:
visco.head()

## Removing erroneous samples

In [ ]:
visco = visco.query("source != 'A81'")
visco = visco.query("source != 'A118'")
visco = visco.query("source != 'A85'")

## Correcting SMILES typos

In [ ]:
def validate_smiles(smiles):
    if pd.isna(smiles):
        return True
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return False
    return True

def correct_smiles(smiles):
    if smiles == 'CCCCCCCCCCCCCCCCOCC(O)CN+(C)C.[Br-]':
        return 'CCCCCCCCCCCCCCCCOCC(O)C[N+](C)C.[Br-]'
    elif smiles == 'CCCCCCCCCCCCCCCCN+(C)C.[Cl-]':
        return 'CCCCCCCCCCCCCCCC[N+](C)C.[Cl-]'
    elif smiles == 'CCCCCCCCCCCCCCCCN+(C)C.[Br-]':
        return 'CCCCCCCCCCCCCCCC[N+](C)C.[Br-]'
    elif smiles == 'CCCCCCCCCCCC(=O)NCCCN+(C)C([O-])=O':
        return 'CCCCCCCCCCCC(=O)NCCC[N+](C)C([O-])=O'
    elif smiles == 'CCCCCCCCCCCC(=O)NCCCN+(C)(C)CC([O-])=O':
        return 'CCCCCCCCCCCC(=O)NCCC[N+](C)(C)CC([O-])=O'
    elif smiles == '[Al+3].[O-]N+[O-].[O-]N+[O-].[O-]N+[O-]':
        return '[Al+3].[O-][N+][O-].[O-][N+][O-].[O-][N+][O-]'
    elif smiles == '[Na+].[O-]N+[O-]':
        return '[Na+].[O-][N+][O-]'
    elif smiles == 'CCCCCC1C(CCCCCC(=O)NCCCN(C)(C)CC(=O)[O-])C(C=CC1CCCCCC)CCCCCC(=O)NCCCN(C)(C)CC(=O)[O-]':
        return 'CCCCCCCCCCCC[N+](C)(C)CC(=O)[O-]'
    return smiles

In [ ]:
visco['smiles_surfactant1'] = visco['smiles_surfactant1'].apply(correct_smiles)
visco['smiles_surfactant2'] = visco['smiles_surfactant2'].apply(correct_smiles)
visco['smiles_additive1'] = visco['smiles_additive1'].apply(correct_smiles)
visco['smiles_additive2'] = visco['smiles_additive2'].apply(correct_smiles)

In [ ]:
smiles = list(visco['smiles_surfactant1'].unique())
smiles.extend(visco['smiles_surfactant2'].unique())
smiles.extend(visco['smiles_additive1'].unique())
smiles.extend(visco['smiles_additive2'].unique())
len(smiles)
for smile in smiles:
    if not validate_smiles(smile):
        print(smile)

In [ ]:
visco.info()

## SMILES to canonical

In [ ]:
def smiles_to_canonical(smiles):
    if pd.isna(smiles):
        return smiles
    return Chem.MolToSmiles(Chem.MolFromSmiles(smiles))

In [ ]:
columns = ['smiles_surfactant1', 'smiles_surfactant2','smiles_additive1', 'smiles_additive2']
visco[columns] = visco[columns].apply(lambda col: col.map(smiles_to_canonical))

In [ ]:
visco.duplicated().sum()

In [ ]:
visco = visco.drop_duplicates()

In [ ]:
save_features(visco, 'viscoelastic_cleaned')